In [1]:
import pandas as pd

df = pd.read_excel(
    "WLS Data.xlsx",
    sheet_name="Worksheet Final",
    keep_default_na=False
)

print(df.columns)

Index(['Bloomberg Title', 'Ticker', 'Exchange Code', 'Equity Exchange Name',
       'MIC', 'Operating MIC', 'MIC Exchange Name', 'Security Type', 'Name',
       'ISO Country', 'Weight', 'Shares', 'Price'],
      dtype='object')


In [2]:
# Preserve all distinct Bloomberg ticker + MIC combinations
listings = (
    df[["Ticker", "MIC"]]
    .assign(
        Ticker=lambda x: x["Ticker"].astype(str).str.strip(),
        MIC=lambda x: x["MIC"].astype(str).str.strip()
    )
)

listings = listings[
    listings["Ticker"].ne("") & listings["MIC"].ne("")
].drop_duplicates(subset=["Ticker", "MIC"])

# Keep a list of ticker strings for code that expects one
tickers = listings["Ticker"].tolist()

print(f"Number of listings: {len(listings):,}")
print(f"Number of ticker entries: {len(tickers):,}")
print(tickers[:10])

Number of listings: 10,420
Number of ticker entries: 10,420
['NVDA', 'AAPL', 'MSFT', 'AMZN', 'GOOGL', '2330', 'AVGO', 'GOOG', 'META', 'TSLA']


In [3]:
mic_to_suffix = {
    # United States
    'XNYS': '', 'XNGS': '', 'XNMS': '', 'XNCM': '', 'XASE': '', 'BATS': '',

    # Asia
    'XTKS': '.T',
    'XNSE': '.NS', 'XBOM': '.BO',
    'XSSC': '.SS', 'XSEC': '.SZ',
    'XHKG': '.HK',
    'ROCO': '.TW', 'XTAI': '.TW',  # Corrected Taiwan suffix
    'XKRX': '.KS', 'XKOS': '.KQ',
    'XIDX': '.JK',
    'XKLS': '.KL',
    'XSES': '.SI',
    'XBKK': '.BK',
    'XPHS': '.PS',

    # Europe
    'XLON': '.L',
    'XSWX': '.SW',
    'XETR': '.DE',
    'XPAR': '.PA',
    'XAMS': '.AS',
    'XSTO': '.ST',
    'XOSL': '.OL',
    'XCSE': '.CO',
    'XHEL': '.HE',
    'XATH': '.AT',
    'XWAR': '.WA',
    'XBRU': '.BR',
    'XWBO': '.VI',
    'XLIS': '.LS',
    'XDUB': '.IR',
    'MTAA': '.MI',

    # Americas
    'XTSE': '.TO',
    'XASX': '.AX',
    'XJSE': '.JO',
    'XSGO': '.SN',
    'XMEX': '.MX',
    'XBOG': '.CL',

    # Middle East and Africa
    'XSAU': '.SR',
    'XTAE': '.TA',
    'XIST': '.IS',
    'XDFM': '.AE',
    'XADS': '.AE',
    'XCAI': '.CA',
    'XKUW': '.KW',
    'DSMD': '.QA',
}

swedish_share_class_tickers = {
    'INVEB', 'ATCOA', 'VOLVB', 'SWEDA', 'ASSAB',
    'SEBA', 'ATCOB', 'ERICB', 'INVEA', 'SHBA',
    'HEXAB', 'SAABB', 'EPIA', 'SECUB', 'LIFCOB',
    'TRELB', 'ADDTB', 'TEL2B', 'SKFB',
    'SKAB', 'EPIB', 'SSABB', 'NIBEB', 'HMB',
    'ESSITYB', 'INDUC', 'BEIJB', 'SCAB',
    'GETIB', 'SECTB', 'SAGAD', 'WALLB', 'ELUXB',
    'NCCB', 'ENGCONB', 'CREDA', 'COREB', 'INTEAD',
    'BALDB', 'EMILB', 'MMGRB', 'BIOAB', 'RATOB',
    'NOLAB', 'PEABB', 'BILIA', 'INDUA', 'VOLVA',
    'SHBB', 'PNDXB', 'LATOB', 'SVEAFB', 'NEWAB',
    'HOLMB', 'BETSB', 'VITB', 'HUSQB', 'MTGB',
    'SSABA', 'INTEAB', 'SDIPB', 'ALLIGOB', 'SBBB',
    'RAYB', 'SAGAB', 'LAGRB', 'EMBRACB', 'SWECB',
    'ALIFB', 'LOGIB', 'ARJOB', 'CLAB', 'MCOVB',
    'LUNDB', 'FPARA', 'SEBC', 'BEIAB', 'STEFB',
    'ATRLJB', 'HUFVA', 'VOLCARB', 'BIOGB', 'EKTAB',
    'HPOLB', 'STORYB', 'VBGB', 'OEMB', 'KINVB',
    'SVOLB', 'BERGB', 'ANODB', 'TRUEB', 'PLAZB',
    'EPROB', 'SKISB', 'SLPB', 'CLASB'
}

danish_share_class_tickers = {
    'ALKB', 'AMBUB', 'CARLB', 'COLOB',
    'HLUNA', 'HLUNB', 'MAERSKA', 'MAERSKB',
    'NOVOB', 'NSISB', 'PAALB', 'ROCKB'
}

ticker_transformations = {
    # Sweden: add a hyphen for verified share-class tickers (e.g. INVEB -> INVE-B)
    'XSTO': lambda ticker: (
        ticker[:-1] + '-' + ticker[-1]
        if ticker in swedish_share_class_tickers
        else ticker
    ),

    # Denmark: add a hyphen for verified share-class tickers (e.g. NOVOB -> NOVO-B)
    'XCSE': lambda ticker: (
        ticker[:-1] + '-' + ticker[-1]
        if ticker in danish_share_class_tickers
        else ticker
    ),
    
    # Hong Kong: 700 -> 0700
    'XHKG': lambda ticker: ticker.zfill(4),

    # UK: remove trailing hyphen (e.g. AV- -> AV)
    'XLON': lambda ticker: (
        ticker[:-1] if ticker.endswith('-') else ticker
    ),

    # Mexico: remove a trailing asterisk (e.g. PE&OLES* -> PE&OLES)
    'XMEX': lambda ticker: (
        ticker[:-1] if ticker.endswith('*') else ticker
    ),
}

# Bloomberg-to-Yahoo ticker overrides for symbols that differ
ticker_overrides = {
    # India
    'HDFCB': 'HDFCBANK.NS',
    'ICICIBC': 'ICICIBANK.NS',
    'BHARTI': 'BHARTIARTL.NS',
    'AXSB': 'AXISBANK.NS',
    'INFO': 'INFY.NS',
    'BAF': 'BAJFINANCE.NS',
    'MM': 'M&M.NS',
    'SHFL': 'SHRIRAMFIN.NS',
    'SUNP': 'SUNPHARMA.NS',
    'TTAN': 'TITAN.NS',
    'BHE': 'BEL.NS',
    'APNT': 'ASIANPAINT.NS',
    'MSIL': 'MARUTI.NS',
    'UTCEM': 'ULTRACEMCO.NS',
    'DIVI': 'DIVISLAB.NS',
    'ADANIGR': 'ADANIGREEN.NS',
    'MAXHEALT': 'MAXHEALTH.NS',
    'MOTHERSO': 'MOTHERSON.NS',
    'BJAUT': 'BAJAJ-AUTO.NS',
    'TVSL': 'TVSMOTOR.NS',
    'LAURUS': 'LAURUSLABS.NS',
    'ADSEZ': 'ADANIPORTS.NS',
    'HUVR': 'HINDUNILVR.NS',
    'TATA': 'TATASTEEL.NS',
    'ADANI': 'ADANIPOWER.NS',
    'HNDL': 'HINDALCO.NS',
    'JSTL': 'JSWSTEEL.NS',
    'ADE': 'ADANIENT.NS',
    'PWGR': 'POWERGRID.NS',
    'BJFIN': 'BAJAJFINSV.NS',
    'EIM': 'EICHERMOT.NS',
    'HCLT': 'HCLTECH.NS',
    'HNAL': 'HAL.NS',
    'NEST': 'NESTLEIND.NS',
    'APHS': 'APOLLOHOSP.NS',
    'COAL': 'COALINDIA.NS',
    'FB': 'FEDERALBNK.NS',
    'CIFC': 'CHOLAFIN.NS',
    'KKC': 'CUMMINSIND.NS',
    'IH': 'INDHOTEL.NS',
    'HMCL': 'HEROMOTOCO.NS',
    'SRCM': 'SHREECEM.NS',
    'IDFCFB': 'IDFCFIRSTB.NS',
    'AL': 'ASHOKLEY.NS',
    'PSYS': 'PERSISTENT.NS',
    'TATACONS': 'TATACONSUM.NS',
    'YES': 'YESBANK.NS',
    'GVTD': 'GVT&D.NS',
    'BRIT': 'BRITANNIA.NS',
    'TPWR': 'TATAPOWER.NS',
    'IIB': 'INDUSINDBK.NS',
    'BHFC': 'BHARATFORG.NS',
    'SOIL': 'SOLARINDS.NS',
    'DRRD': 'DRREDDY.NS',
    'INDUSTOW': 'INDUSTOWER.NS',
    'INFOE': 'NAUKRI.NS',
    'BJHI': 'BAJAJHLDNG.NS',
    'LPC': 'LUPIN.NS',
    'SUEL': 'SUZLON.NS',
    'POWF': 'PFC.NS',
    'ARBP': 'AUROPHARMA.NS',
    'DLFU': 'DLF.NS',
    'WPRO': 'WIPRO.NS',
    'ADANIENS': 'ADANIENSOL.NS',
    'JINDALST': 'JINDALSTEL.NS',
    'PIDI': 'PIDILITIND.NS',
    'POWERIND': 'POWERINDIA.NS',
    'MRCO': 'MARICO.NS',
    'PHNX': 'PHOENIXLTD.NS',
    'POLICYBZ': 'POLICYBZR.NS',
    'IOCL': 'IOC.NS',
    'APAT': 'APLAPOLLO.NS',
    'BOB': 'BANKBARODA.NS',
    'FORH': 'FORTIS.NS',
    'MAXF': 'MFSL.NS',
    'RECL': 'RECLTD.NS',
    'ABCAP': 'ABCAPITAL.NS',
    'RDCK': 'RADICO.NS',
    'GNP': 'GLENMARK.NS',
    'HPCL': 'HINDPETRO.NS',
    'GCPL': 'GODREJCP.NS',
    'ATHERENE': 'ATHERENERG.NS',
    'BOS': 'BOSCHLTD.NS',
    'SIEM': 'SIEMENS.NS',
    'APR': 'APARINDS.NS',
    'GMRAIRPO': 'GMRAIRPORT.NS',
    'KVB': 'KARURVYSYA.NS',
    'SUF': 'SUNDARMFIN.NS',
    'WLCO': 'WELCORP.NS',
    'MUTH': 'MUTHOOTFIN.NS',
    'NACL': 'NATIONALUM.NS',
    'HMFC': 'HFCL.NS',
    'NFIL': 'NAVINFLUOR.NS',
    'UPLL': 'UPL.NS',
    'INBK': 'INDIANB.NS',
    'KEII': 'KEI.NS',
    'MPHL': 'MPHASIS.NS',
    'UNBK': 'UNIONBANK.NS',
    'JDSL': 'JSL.NS',
    'RBK': 'RBLBANK.NS',
    'OINL': 'OIL.NS',
    'REDI': 'REDINGTON.NS',
    'JSW': 'JSWENERGY.NS',
    'BIOS': 'BIOCON.NS',
    'PEPL': 'PRESTIGE.NS',
    'ACEM': 'AMBUJACEM.NS',
    'ZYDUSLIF': 'ZYDUSLIFE.NS',
    'PLNG': 'PETRONET.NS',
    'PIRAMALF': 'PIRAMALFIN.NS',
    'TPW': 'TORNTPOWER.NS',
    'HAVL': 'HAVELLS.NS',
    'VOLT': 'VOLTAS.NS',
    'KALYANKJ': 'KALYANKJIL.NS',
    'SI': 'SUPREMEIND.NS',
    'CLGT': 'COLPAL.NS',
    'CUBK': 'CUB.NS',
    'IPCA': 'IPCALAB.NS',
    'OBER': 'OBEROIRLTY.NS',
    'BLSTR': 'BLUESTARCO.NS',
    'FLUOROCH': 'FLUOROCHEM.NS',
    'ANANDRAT': 'ANANDRATHI.NS',
    'PNBHOUSI': 'PNBHOUSING.NS',
    'DELHIVER': 'DELHIVERY.NS',
    'GPL': 'GODREJPROP.NS',
    'KOEL': 'KIRLOSENG.NS',
    'TCOM': 'TATACOMM.NS',
    'JKCE': 'JKCEMENT.NS',
    'CRIN': 'COROMANDEL.NS',
    'LGEL': 'LGEINDIA.NS',
    'TDPS': 'TDPOWERSYS.NS',
    'JUBI': 'JUBLFOOD.NS',
    'WAAREEEN': 'WAAREEENER.NS',
    'NLL': 'NEULANDLAB.NS',
    'IPRU': 'ICICIPRULI.NS',
    'PIRPHARM': 'PPLPHARMA.NS',
    'NAM': 'NAM-INDIA.NS',
    'WPL': 'WOCKPHARMA.NS',
    'MGFL': 'MANAPPURAM.NS',
    'MOFS': 'MOTILALOFS.NS',
    'EXID': 'EXIDEIND.NS',
    'MMFS': 'M&MFIN.NS',
    'AJP': 'AJANTPHARM.NS',
    'GUJENERG': 'GUJENERGY.NS',
    'CUPD': 'CUPID.NS',
    'APTY': 'APOLLOTYRE.NS',
    'SCHFL': 'SCHAEFFLER.NS',
    'LICHF': 'LICHSGFIN.NS',
    'GESCO': 'GESHIP.NS',
    'BANDHAN': 'BANDHANBNK.NS',
    'ITCHOTEL': 'ITCHOTELS.NS',
    'AIAE': 'AIAENG.NS',
    'POONAWAL': 'POONAWALLA.NS',
    'ASTRA': 'ASTRAL.NS',
    'HSCH': 'HSCL.NS',
    'HCP': 'HINDCOPPER.NS',
    'DLPL': 'LALPATHLAB.NS',
    'CRAFTSMA': 'CRAFTSMAN.NS',
    'JSAW': 'JINDALSAW.NS',
    'CHOLAHLD': 'CHOLAHLDNG.NS',
    'TMX': 'THERMAX.NS',
    'PREMIERE': 'PREMIERENE.NS',
    'DALBHARA': 'DALBHARAT.NS',
    'MAZDOCKS': 'MAZDOCK.NS',
    'BOMH': 'MAHABANK.NS',
    'STARHEAL': 'STARHEALTH.NS',
    'UJJIVANS': 'UJJIVANSFB.NS',
    'KPR': 'KPRMILL.NS',
    'ASTM': 'ASTRAMICRO.NS',
    'USM': 'USHAMART.NS',
    'ELEQ': 'ELGIEQUIP.NS',
    'GRAN': 'GRANULES.NS',
    'KBL': 'KTKBANK.NS',
    'GP': 'GODFRYPHLP.NS',
    'BRGR': 'BERGEPAINT.NS',
    'LIIL': 'LINDEINDIA.NS',
    'BOI': 'BANKINDIA.NS',
    'TMKN': 'TIMKEN.NS',
    'CCRI': 'CONCOR.NS',
    'BHARTIHE': 'BHARTIHEXA.NS',
    'GLXO': 'GLAXO.NS',
    'NARH': 'NH.NS',
    'SAMMAANC': 'SAMMAANCAP.NS',
    'KJC': 'KAJARIACER.NS',
    'TRCL': 'RAMCOCEM.NS',
    'ONESOURC': 'ONESOURCE.NS',
    'ZCVCS': 'ZFCVINDIA.NS',
    'FNXC': 'FINCABLES.NS',
    'DATAPATT': 'DATAPATTNS.NS',
    'HOMEFIRS': 'HOMEFIRST.NS',
    'SLPA': 'SHILPAMED.NS',
    'BRGD': 'BRIGADE.NS',
    'COCHIN': 'COCHINSHIP.NS',
    'GWN': 'GRINDWELL.NS',
    'TELX': 'TATAELXSI.NS',
    'DN': 'DEEPAKNTR.NS',
    'DFPC': 'DEEPAKFERT.NS',
    'VATW': 'WABAG.NS',
    'CYL': 'CYIENT.NS',
    'ARTO': 'AARTIIND.NS',
    'AISG': 'ASAHIINDIA.NS',
    'ATLP': 'ATUL.NS',
    'TATAINVE': 'TATAINVEST.NS',
    'ACMESOLA': 'ACMESOLAR.NS',
    'UBBL': 'UBL.NS',
    'GABR': 'GABRIEL.NS',
    '3M': '3MINDIA.NS',
    'HBLENGIN': 'HBLENGINE.NS',
    'NLC': 'NLCINDIA.NS',
    'SHADOWF': 'SHADOWFAX.NS',
    'LLOYDSEN': 'LLOYDSENGG.NS',
    'ENGR': 'ENGINERSIN.NS',
    'ARCP': 'ANANTRAJ.NS',
    'PATANJAL': 'PATANJALI.NS',
    'FML': 'FORCEMOT.NS',
    'TLNGR': 'TI.NS',
    'TTCH': 'TATACHEM.NS',
    'ARENM': 'ARE&M.NS',
    'BLUESTON': 'BLUESTONE.NS',
    'ARVND': 'ARVIND.NS',
    'CSTRL': 'CASTROLIND.NS',
    'INXW': 'INOXWIND.NS',
    'ECLX': 'ECLERX.NS',
    'EDEL': 'EDELWEISS.NS',
    'BRCM': 'BALRAMCHIN.NS',
    'PCJL': 'PCJEWELLER.NS',
    'SHEP': 'SHAILY.NS',
    'NTCPH': 'NATCOPHARM.NS',
    'VTEX': 'VTL.NS',
    'CCLP': 'CCL.NS',
    'FSOL': 'FSL.NS',
    'JUBLPHAR': 'JUBLPHARMA.NS',
    'VAMP': 'VOLTAMP.NS',
    'ZEN': 'ZENTEC.NS',
    'JPVL': 'JPPOWER.NS',
    'HWA': 'HONAUT.NS',
    'SHYAMMET': 'SHYAMMETL.NS',
    'MRKS': 'MARKSANS.NS',
    'AADHARHF': 'AADHARHFC.NS',
    'EQUITASB': 'EQUITASBNK.NS',
    'CEAT': 'CEATLTD.NS',
    'TRIV': 'TRITURBINE.NS',
    'CREDAG': 'CREDITACC.NS',
    'EID': 'EIDPARRY.NS',
    'SHRIPIST': 'SHRIPISTON.NS',
    'CHMB': 'CHAMBLFERT.NS',
    'WELSPUNL': 'WELSPUNLIV.NS',
    'HTSMF': 'HATSUN.NS',
    'UTIAM': 'UTIAMC.NS',
    'BLACKBUC': 'BLACKBUCK.NS',
    'SYNG': 'SYNGENE.NS',
    'SCHI': 'SUDARSCHEM.NS',
    'ENDU': 'ENDURANCE.NS',
    'WHIRL': 'WHIRLPOOL.NS',
    'CFIS': 'CHOICEIN.NS',
    'RMT': 'RATNAMANI.NS',
    'GRWRHITE': 'GRWRHITECH.NS',
    'PLM': 'POLYMED.NS',
    'HAPPYFOR': 'HAPPYFORGE.NS',
    'CANF': 'CANFINHOME.NS',
    'EIH': 'EIHOTEL.NS',
    'TIME': 'TIMETECHNO.NS',
    'RMKF': 'RKFORGE.NS',
    'ZENT': 'ZENSARTECH.NS',
    'METROHL': 'METROPOLIS.NS',
    'KECI': 'KEC.NS',
    'VGRD': 'VGUARD.NS',
    'GRIL': 'GRAPHITE.NS',
    'RPWR': 'RPOWER.NS',
    'LEMONTRE': 'LEMONTREE.NS',
    'CLPL': 'CAPLIPOINT.NS',
    'SWH': 'JSWHL.NS',
    'JKBK': 'J&KBANK.NS',
    'TJL': 'THANGAMAYL.NS',
    'LMAX': 'LUMAXTECH.NS',
    'SPPT': 'SPLPETRO.NS',
    'MHSC': 'MAHSCOOTER.NS',
    'PFIZ': 'PFIZER.NS',
    'JUBLINGR': 'JUBLINGREA.NS',
    'DCBB': 'DCBBANK.NS',
    'MAHGL': 'MGL.NS',
    'NJCC': 'NCC.NS',
    'DYTC': 'DYNAMATECH.NS',
    'TFCI': 'TFCILTD.NS',
    'KKPC': 'KIRLPNU.NS',
    'ACCE': 'ACE.NS',
    'ZYWL': 'ZYDUSWELL.NS',
    'GODPI': 'GPIL.NS',
    'PRICOL': 'PRICOLLTD.NS',
    'GILL': 'GILLETTE.NS',
    'INMART': 'INDIAMART.NS',
    'RINDL': 'RAIN.NS',
    'INOXINDI': 'INOXINDIA.NS',
    'GPPV': 'GPPL.NS',
    'INDA': 'INTELLECT.NS',
    'PRIF': 'PFOCUS.NS',
    'SIGNATUR': 'SIGNATURE.NS',
    'SEML': 'SARDAEN.NS',
    'ELCN': 'ELECON.NS',
    'BYRCS': 'BAYERCROP.NS',
    'GRAV': 'GRAVITA.NS',
    'GDSP': 'GODREJIND.NS',
    'GMM': 'GMMPFAUDLR.NS',
    'AARTIPHA': 'AARTIPHARM.NS',
    'SARE': 'SAREGAMA.NS',
    'CONCORDB': 'CONCORDBIO.NS',
    'ALPM': 'APLLTD.NS',
    'IBULLSLT': 'IBULLSLTD.NS',
    'SSOF': 'SONATSOFTW.NS',
    'KKB': 'KIRLOSBROS.NS',
    'AEGISVOP': 'AEGISVOPAK.NS',
    'GEXP': 'GOKEX.NS',
    'JKI': 'JKTYRE.NS',
    'CPBI': 'CENTURYPLY.NS',
    'RELG': 'RELIGARE.NS',
    'GMDC': 'GMDCLTD.NS',
    'GTFL': 'GARFIBRES.NS',
    'FNXP': 'FINPIPE.NS',
    'AXET': 'AXISCADES.NS',
    'GPIN': 'GENUSPOWER.NS',
    'CYIENTDL': 'CYIENTDLM.NS',
    'LLOYDSET': 'LLOYDSENT.NS',
    'GMRPUI': 'GMRP&UI.NS',
    'ION': 'IONEXCHANG.NS',
    'VI': 'VESUVIUS.NS',
    'PRJ': 'PRAJIND.NS',
    'JSWCEMEN': 'JSWCEMENT.NS',
    'PWI': 'PRECWIRE.NS',
    'ICNT': 'ICIL.NS',
    'MHS': 'MAHSEAMLES.NS',
    'AGARWALE': 'AGARWALEYE.NS',
    'CBOI': 'CENTRALBK.NS',
    'TXMRE': 'TEXRAIL.NS',
    'SKF': 'SKFINDIA.NS',
    'ELSC': 'ELECTCAST.NS',
    'LGBB': 'LGBBROSLTD.NS',
    'TIMX': 'TIMEX.BO',
    'ARVINDFA': 'ARVINDFASN.NS',
    'EPLL': 'EPL.NS',
    'KNPL': 'KANSAINER.NS',
    'KKF': 'KIRLFER.BO',
    'PTCIN': 'PTC.NS',
    'BECTORS': 'BECTORFOOD.NS',
    'THYROCAR': 'THYROCARE.NS',
    'POWM': 'POWERMECH.NS',
    'BLSIN': 'BLS.NS',
    'BATA': 'BATAINDIA.NS',
    'METROBRA': 'METROBRAND.NS',
    'VO': 'VINATIORGA.NS',
    'KRB': 'KRBL.NS',
    'JKLC': 'JKLAKSHMI.NS',
    'BLIS': 'BLISSGVS.NS',
    'SVLS': 'SUVEN.NS',
    'SUDEEPPH': 'SUDEEPPHRM.NS',
    'KPIG': 'KPIGREEN.NS',
    'NGE': 'JAYNECOIND.NS',
    'GOAGRO': 'GODREJAGRO.NS',
    'INDIASHL': 'INDIASHLTR.NS',
    'JMNA': 'JAMNAAUTO.NS',
    'MAXESTAT': 'MAXESTATES.NS',
    'TIPSMUSI': 'TIPSMUSIC.NS',
    'VREL': 'V2RETAIL.NS',
    'ASKAUTOL': 'ASKAUTOLTD.NS',
    'ASTR': 'ASTRAZEN.NS',
    'TRID': 'TRIDENT.NS',
    'BJH': 'BAJAJHIND.NS',
    'KMC': 'KOVAI.BO',
    'AACL': 'ALKYLAMINE.NS',
    'BDE': 'BLUEDART.NS',
    'DCMS': 'DCMSHRIRAM.NS',
    'BNCO': 'BANCOINDIA.NS',
    'RRW': 'RAMRAT.NS',
    'MAST': 'MASTEK.NS',
    'SKPI': 'SHAKTIPUMP.NS',
    'JYL': 'JYOTHYLAB.NS',
    'NRBBR': 'NRBBEARING.NS',
    'IGSEC': 'ISGEC.NS',
    'HAPPSTMN': 'HAPPSTMNDS.NS',
    'BALUFORG': 'BALUFORGE.NS',
    'BORORENE': 'BORORENEW.NS',
    'ASMN': 'ASHAPURMIN.NS',
    'ICP': 'IOLCP.NS',
    'SANOFICO': 'SANOFI.NS',
    'EUREKAFO': 'EUREKAFORB.NS',
    'ROLEXRIN': 'ROLEXRINGS.NS',
    'FTXC': 'FCL.NS',
    'KNM': 'KENNAMET.NS',
    'INDIGOPN': 'INDIGOPNTS.NS',
    'RAYMONDR': 'RAYMONDREL.NS',
    'GALSURF': 'GALAXYSURF.NS',
    'AVTS': 'AVANTEL.NS',
    'VIP': 'VIPIND.NS',
    'GAEX': 'GAEL.NS',
    'BCORP': 'BIRLACORPN.NS',
    'SYR': 'SURYAROSNI.NS',
    'SANL': 'SANOFI.NS',
    'TRAVELFO': 'TRAVELFOOD.NS',
    'PRSMJ': 'PRSMJOHNSN.NS',
    'GLIN': 'GOODLUCK.NS',
    'RITE': 'RITES.NS',
    'FIEM': 'FIEMIND.NS',
    'FDCLT': 'FDC.NS',
    'MBAP': 'MBAPL.NS',
    'WAREERTL': 'WAAREERTL.NS',
    'GRV': 'GREAVESCOT.NS',
    'OPIL': 'OPTIEMUS.NS',
    'WESL': 'WEBELSOLAR.NS',
    'ORIENTEL': 'ORIENTELEC.NS',
    'INOXGREE': 'INOXGREEN.NS',
    'PARKHOSP': 'PARKHOSPS.NS',
    'SISLTD': 'SHAREINDIA.NS',
    'VIKRAMSO': 'VIKRAMSOLR.NS',
    'TANF': 'TANFACIND.BO',
    'IDEAFORG': 'IDEAFORGE.NS',
    'VRLL': 'VRLLOG.NS',
    'TRPC': 'TCI.NS',
    'SWE': 'SWARAJENG.NS',
    'SHMO': 'SHARDAMOTR.NS',
    'NETM': 'NETWORK18.NS',
    'GLDM': 'GOLDIAM.NS',
    'TRANSRAI': 'TRANSRAILL.NS',
    'UFLX': 'UFLEX.NS',
    'ATLANTAE': 'ATLANTAELE.NS',
    'INNOVACA': 'INNOVACAP.NS',
    'SHRS': 'RENUKA.NS',
    'SUNF': 'SUNFLAG.NS',
    'SPADV': 'SPARC.NS',
    'PCAI': 'PICCADIL.NS',
    'GOLI': 'GULFOILLUB.NS',
    'RTPOW': 'RTNPOWER.NS',
    'PWRL': 'POWERICA.NS',
    'RALI': 'RALLIS.NS',
    'KS': 'KSL.NS',
    'ICEM': 'INDIACEM.NS',
    'AUPS': 'AURIONPRO.NS',
    'RPE': 'RPEL.NS',
    'TTKPT': 'TTKPRESTIG.NS',
    'TTLS': 'TTML.NS',
    'STEC': 'SOTL.NS',
    'SHCR': 'SHARDACROP.NS',
    'VSTT': 'VSTTILLERS.NS',
    'MINF': 'MANINFRA.NS',
    'VDI': 'VADILALIND.NS',
    'KNRC': 'KNRCON.NS',
    'BANSALWI': 'BANSALWIRE.NS',
    'SYIL': 'STYLAMIND.NS',
    'TC': 'THOMASCOOK.NS',
    'AHLU': 'AHLUCONT.NS',
    'SECIS': 'SIS.NS',
    'GRLM': 'GREENLAM.NS',
    'ORCP': 'ORCHPHARMA.NS',
    'JBIL': 'JAIBALAJI.NS',
    'ARTD': 'AARTIDRUGS.NS',
    'REFX': 'REFEX.NS',
    'PNCL': 'PNCINFRA.NS',
    'AGIG': 'AGI.NS',
    'UVC': 'UNIVCABLES.NS',
    'MAPMYIND': 'MAPMYINDIA.NS',
    'RAYMONDL': 'RAYMONDLSL.NS',
    'JUST': 'JUSTDIAL.NS',
    'RLXF': 'RELAXO.NS',
    'SRIN': 'SUNTECK.NS',
    'NIRL': 'NIRLON.BO',
    'GALK': 'GUJALKALI.NS',
    'SUBR': 'SUBROS.NS',
    'WPI': 'WPIL.BO',
    'LATENTVI': 'LATENTVIEW.NS',
    'IPMC': 'INDRAMEDCO.NS',
    'BJE': 'BAJAJELEC.NS',
    'BOSCHHCI': 'BOSCH-HCIL.NS',
    'CHEMPLAS': 'CHEMPLASTS.NS',
    'ASBL': 'ASHOKA.NS',
    'SHREEJIS': 'SHREEJISPG.NS',
    'ORKLAIND': 'ORKLAINDIA.NS',
    'SCTE': 'SHILCTECH.NS',
    'ADVENZY': 'ADVENZYMES.NS',
    'SMARTWOR': 'SMARTWORKS.NS',
    'CAPILLAR': 'CAPILLARY.NS',
    'LUMX': 'LUMAXIND.NS',
    'RUSTOMJE': 'RUSTOMJEE.NS',
    'RPGL': 'RPGLIFE.NS',
    'MPS': 'MPSLTD.NS',
    'VGM': 'VAIBHAVGBL.NS',
    'GANESHHO': 'GANESHHOU.NS',
    'DAGRI': 'DHANUKA.NS',
    'IFBI': 'IFBIND.NS',
    'DPAL': 'DPABHUSHAN.NS',
    'GW': 'GRAUWEIL.BO',
    'RESP': 'RESPONIND.NS',
    'SG': 'SHANTIGEAR.NS',
    'ALOK': 'ALOKINDS.NS',
    'COCKERIL': 'COCKERILL.BO',
    'PACD': 'PACEDIGITK.NS',
    'KALPATAR': 'KALPATARU.NS',
    'MHRL': 'MHRIL.NS',
    'ARTMSL': 'ARTEMISMED.NS',
    'NELI': 'NAVNETEDUL.NS',
    'IGLY': 'INDIAGLYCO.NS',
    'SYML': 'SYMPHONY.NS',
    'DATAMATI': 'DATAMATICS.NS',
    'MIDWESTL': 'MIDWESTLTD.NS',
    'KKIL': 'KIRLOSIND.NS',
    'KPDL': 'KOLTEPATIL.NS',
    'PSPPL': 'PSPPROJECT.NS',
    'RATI': 'RTNINDIA.NS',
    'MUTHOOTM': 'MUTHOOTMF.NS',
    'RSYS': 'RSYSTEMS.NS',
    'SAATVIK': 'SAATVIKGL.NS',
    'DBCL': 'DBCORP.NS',
    'KEKC': 'KKCL.NS',
    'JTEKT': 'JTEKTINDIA.NS',
    'OSWALPUM': 'OSWALPUMPS.NS',
    'SINDHUTR': 'SINDHUTRAD.NS',

    
    # Saudi Arabia
    'RJHI': '1120.SR',
    'ARAMCO': '2222.SR',
    'SNB': '1180.SR',
    'MAADEN': '1211.SR',
    'STC': '7010.SR',
    'ALINMA': '1150.SR',
    'SAFCO': '2020.SR',
    'EEC': '7020.SR',
    'ALMARAI': '2280.SR',
    'RIBL': '1010.SR',
    'SABIC': '2010.SR',
    'ACWA': '2082.SR',
    'BSF': '1050.SR',
    

    # Taiwan — Taipei Exchange
    '5274': '5274.TWO',
    '3105': '3105.TWO',
    '6274': '6274.TWO',
    '3081': '3081.TWO',
    '6488': '6488.TWO',
    '8299': '8299.TWO',
    '6223': '6223.TWO',
    '3529': '3529.TWO',
    '5347': '5347.TWO',

    # UAE
    'EMIRATES': 'EMIRATESNBD.AE',

    # Ireland
    'AIBG': 'A5G.IR',

    # Finland
    'NDA': 'NDA-FI.HE',

    # Malaysia
    'MAY': '1155.KL',
    'PBK': '1295.KL',

    # Singapore
    'DBS': 'D05.SI',
    'OCBC': 'O39.SI',
    'SGX': 'S68.SI',
    'ST': 'Z74.SI',
    'UOB': 'U11.SI',

    # Chile
    'FALAB': 'FALABELLA.SN',

    # Sweden
    'OCTVSDB': 'OCTV-SDB.ST',
    'ASMDEE': 'ASMDEE-B.ST',
}

In [4]:
# Create mapping DataFrame, keeping one row per Bloomberg ticker + MIC
mapping = (
    df[["Ticker", "MIC", "ISO Country", "Name"]]
    .copy()
)

for col in ["Ticker", "MIC", "ISO Country", "Name"]:
    mapping[col] = mapping[col].astype(str).str.strip()

mapping = mapping[
    mapping["Ticker"].ne("") & mapping["MIC"].ne("")
].drop_duplicates(subset=["Ticker", "MIC"]).copy()

# Generate Yahoo Finance ticker and mapping status
def map_ticker(row):
    ticker = str(row["Ticker"]).strip().replace("/", "-")
    mic = str(row["MIC"]).strip()
    country = str(row["ISO Country"]).strip()

    if ticker in ticker_overrides:
        return ticker_overrides[ticker], "Ticker override"

    if mic in ticker_transformations:
        ticker = ticker_transformations[mic](ticker)

    if mic in mic_to_suffix:
        return ticker + mic_to_suffix[mic], "MIC rule"

    # Country-based fallback for securities with missing MIC
    country_suffix = {
        "ES": ".MC",
        "BR": ".SA",
    }

    if mic in ("0", "nan", "") and country in country_suffix:
        return ticker + country_suffix[country], "Country fallback"

    return None, "Needs review"


mapping[["YF_Ticker", "Mapping_Status"]] = mapping.apply(
    lambda row: pd.Series(map_ticker(row)),
    axis=1
)

display(mapping.head(10))
print(mapping["Mapping_Status"].value_counts())
print(f"Total listings in mapping: {len(mapping):,}")

,Ticker,MIC,ISO Country,Name,YF_Ticker,Mapping_Status
0,NVDA,XNGS,US,NVIDIA Corp,NVDA,MIC rule
1,AAPL,XNGS,US,Apple Inc,AAPL,MIC rule
2,MSFT,XNGS,US,Microsoft Corp,MSFT,MIC rule
3,AMZN,XNGS,US,Amazon.com Inc,AMZN,MIC rule
4,GOOGL,XNGS,US,Alphabet Inc,GOOGL,MIC rule
5,2330,ROCO,TW,Taiwan Semiconductor Manufactu,2330.TW,MIC rule
6,AVGO,XNGS,US,Broadcom Inc,AVGO,MIC rule
7,GOOG,XNGS,US,Alphabet Inc,GOOG,MIC rule
8,META,XNGS,US,Meta Platforms Inc,META,MIC rule
9,TSLA,XNGS,US,Tesla Inc,TSLA,MIC rule


Mapping_Status
MIC rule            9616
Ticker override      563
Country fallback     207
Needs review          34
Name: count, dtype: int64
Total listings in mapping: 10,420


In [5]:
import numpy as np
import pandas as pd
import yfinance as yf
import time

all_tickers = sorted(
    mapping.loc[mapping['YF_Ticker'].notna(), 'YF_Ticker']
    .astype(str)
    .str.strip()
    .loc[lambda s: s.ne('') & s.ne('nan')]
    .unique()
)

batches = [
    batch.tolist()
    for batch in np.array_split(all_tickers, 5)
]

for i, batch in enumerate(batches, start=1):
    print(f"Batch {i}: {len(batch):,} tickers")

def test_batch(batch_number):
    if batch_number not in range(1, 6):
        raise ValueError("batch_number must be between 1 and 5")

    batch = batches[batch_number - 1]

    print(f"Testing batch {batch_number}/5 ({len(batch):,} tickers)...")

    data = yf.download(
        tickers=batch,
        period="2d",
        interval="1d",
        auto_adjust=True,
        progress=False,
        threads=False,
        timeout=30
    )

    if data.empty:
        print("Yahoo returned no data. This may be rate-limiting.")
        return None, None

    close = data["Close"]

    # Handle a single ticker separately if necessary
    if isinstance(close, pd.Series):
        close = close.to_frame(name=batch[0])

    results = pd.DataFrame({
        "YF_Ticker": batch
    })

    results["Has_Data"] = results["YF_Ticker"].apply(
        lambda ticker: (
            ticker in close.columns
            and close[ticker].notna().any()
        )
    )

    successes = int(results["Has_Data"].sum())
    failures = len(results) - successes

    print(f"Successful: {successes:,}")
    print(f"Failed:     {failures:,}")
    print(f"Success rate: {successes / len(results) * 100:.2f}%")

    return results, results.loc[~results["Has_Data"]].copy()



Batch 1: 2,074 tickers
Batch 2: 2,074 tickers
Batch 3: 2,074 tickers
Batch 4: 2,073 tickers
Batch 5: 2,073 tickers


In [6]:
print(f"Original Excel rows: {len(df):,}")
print(f"Distinct Bloomberg listings: {len(listings):,}")
print(f"Rows in mapping: {len(mapping):,}")
print(f"Distinct Bloomberg ticker strings: {df['Ticker'].nunique():,}")
print(
    "Distinct Yahoo Finance symbols:",
    mapping.loc[
        mapping["YF_Ticker"].notna() & mapping["YF_Ticker"].ne(""),
        "YF_Ticker"
    ].nunique()
)

print("\nRepeated Bloomberg ticker strings:")
print(
    mapping[mapping["Ticker"].duplicated(keep=False)]
    .sort_values("Ticker")
    [["Ticker", "MIC", "Name", "YF_Ticker"]]
)

Original Excel rows: 10,420
Distinct Bloomberg listings: 10,420
Rows in mapping: 10,420
Distinct Bloomberg ticker strings: 9,850
Distinct Yahoo Finance symbols: 10368

Repeated Bloomberg ticker strings:
       Ticker   MIC                            Name      YF_Ticker
3544   000100  XKRX                      Yuhan Corp      000100.KS
2704   000100  XSEC       TCL Technology Group Corp      000100.SZ
5157   000155  XKRX                   Doosan Co Ltd      000155.KS
6648   000155  XSEC  Sichuan New Energy Power Co Lt      000155.SZ
9390   000157  XKRX                   Doosan Co Ltd      000157.KS
...       ...   ...                             ...            ...
4543      WPL  XNSE                   Wockhardt Ltd  WOCKPHARMA.NS
7329        Z  XNSE   Zee Entertainment Enterprises           Z.NS
2554        Z  XNGS                Zillow Group Inc              Z
3013      ZIM  XNYS  ZIM Integrated Shipping Servic            ZIM
10062     ZIM  XASX           Zimplats Holdings Ltd         

In [7]:
# Find Yahoo Finance symbols assigned to multiple Bloomberg listings
shared_yf = mapping[
    mapping["YF_Ticker"].notna()
    & mapping["YF_Ticker"].astype(str).str.strip().ne("")
    & mapping["YF_Ticker"].duplicated(keep=False)
].sort_values(["YF_Ticker", "MIC", "Ticker"])

print(f"Rows involved in shared Yahoo symbols: {len(shared_yf)}")
print(f"Distinct shared Yahoo symbols: {shared_yf['YF_Ticker'].nunique()}")

display(
    shared_yf[
        ["Ticker", "MIC", "ISO Country", "Name", "YF_Ticker", "Mapping_Status"]
    ]
)

Rows involved in shared Yahoo symbols: 35
Distinct shared Yahoo symbols: 17


,Ticker,MIC,ISO Country,Name,YF_Ticker,Mapping_Status
2006,3105,ROCO,TW,Win Semiconductors Corp,3105.TWO,Ticker override
3686,3105,XTKS,JP,Nisshinbo Holdings Inc,3105.TWO,Ticker override
7510,STC,XKUW,KW,Kuwait Telecommunications Co,7010.SR,Ticker override
4863,STC,XNYS,US,Stewart Information Services C,7010.SR,Ticker override
793,STC,XSAU,SA,Saudi Telecom Co,7010.SR,Ticker override
9830,ARTO,XIDX,ID,Bank Jago Tbk PT,AARTIIND.NS,Ticker override
6274,ARTO,XNSE,IN,Aarti Industries Ltd,AARTIIND.NS,Ticker override
3275,APR,XNSE,IN,Apar Industries Ltd,APARINDS.NS,Ticker override
7794,APR,XWAR,PL,Auto Partner SA,APARINDS.NS,Ticker override
4926,ASTRA,XNSE,IN,Astral Ltd,ASTRAL.NS,Ticker override


In [8]:
import pandas as pd
import yfinance as yf
import time
from pathlib import Path

# Select Batch 1
batch = batches[0]

print(f"Starting Batch 1: {len(batch):,} tickers")
start_time = time.monotonic()

price_series = []
failed_tickers = []

for i, ticker in enumerate(batch, start=1):
    try:
        history = yf.Ticker(ticker).history(
            period="2y",
            interval="1d",
            auto_adjust=True,
            timeout=30
        )

        if not history.empty and history["Close"].notna().any():
            prices = history["Close"].dropna().rename(ticker)
            price_series.append(prices)
        else:
            failed_tickers.append(ticker)

    except Exception:
        failed_tickers.append(ticker)

    # Print progress every 50 tickers and at the end
    if i % 50 == 0 or i == len(batch):
        elapsed = (time.monotonic() - start_time) / 60

        print(
            f"Processed {i:,}/{len(batch):,} | "
            f"Successful: {len(price_series):,} | "
            f"Failed: {len(failed_tickers):,} | "
            f"Elapsed: {elapsed:.1f} minutes"
        )

# Combine successful price histories into one DataFrame
if price_series:
    batch1_prices = pd.concat(price_series, axis=1).sort_index()
    batch1_prices.index.name = "Date"

    # Save the data
    output_path = Path("data/price_data/batch1_prices.parquet")
    output_path.parent.mkdir(parents=True, exist_ok=True)

    batch1_prices.to_parquet(output_path)

    print("\nDownload complete.")
    print(f"Saved to: {output_path}")
    print(f"Tickers with price data: {batch1_prices.shape[1]:,}")
    print(f"Trading dates: {batch1_prices.shape[0]:,}")
    print(f"Failed tickers: {len(failed_tickers):,}")
    print(f"Total elapsed time: {(time.monotonic() - start_time) / 60:.1f} minutes")

else:
    print("No price data was retrieved. No Parquet file was saved.")


Starting Batch 1: 2,074 tickers
Processed 50/2,074 | Successful: 50 | Failed: 0 | Elapsed: 0.4 minutes
Processed 100/2,074 | Successful: 100 | Failed: 0 | Elapsed: 0.8 minutes
Processed 150/2,074 | Successful: 150 | Failed: 0 | Elapsed: 1.1 minutes
Processed 200/2,074 | Successful: 200 | Failed: 0 | Elapsed: 1.5 minutes
Processed 250/2,074 | Successful: 250 | Failed: 0 | Elapsed: 1.9 minutes
Processed 300/2,074 | Successful: 300 | Failed: 0 | Elapsed: 2.2 minutes
Processed 350/2,074 | Successful: 350 | Failed: 0 | Elapsed: 2.6 minutes
Processed 400/2,074 | Successful: 400 | Failed: 0 | Elapsed: 2.9 minutes
Processed 450/2,074 | Successful: 450 | Failed: 0 | Elapsed: 3.3 minutes
Processed 500/2,074 | Successful: 500 | Failed: 0 | Elapsed: 3.6 minutes
Processed 550/2,074 | Successful: 550 | Failed: 0 | Elapsed: 3.9 minutes
Processed 600/2,074 | Successful: 600 | Failed: 0 | Elapsed: 4.3 minutes
Processed 650/2,074 | Successful: 650 | Failed: 0 | Elapsed: 4.6 minutes
Processed 700/2,074 |

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: 1785.TW"}}}
$1785.TW: No data found, symbol may be delisted


Processed 1,100/2,074 | Successful: 1,099 | Failed: 1 | Elapsed: 7.8 minutes


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: 1815.TW"}}}
$1815.TW: No data found, symbol may be delisted


Processed 1,150/2,074 | Successful: 1,148 | Failed: 2 | Elapsed: 8.3 minutes
Processed 1,200/2,074 | Successful: 1,198 | Failed: 2 | Elapsed: 8.7 minutes
Processed 1,250/2,074 | Successful: 1,248 | Failed: 2 | Elapsed: 9.0 minutes
Processed 1,300/2,074 | Successful: 1,298 | Failed: 2 | Elapsed: 9.4 minutes
Processed 1,350/2,074 | Successful: 1,348 | Failed: 2 | Elapsed: 9.8 minutes
Processed 1,400/2,074 | Successful: 1,398 | Failed: 2 | Elapsed: 10.2 minutes
Processed 1,450/2,074 | Successful: 1,448 | Failed: 2 | Elapsed: 10.6 minutes
Processed 1,500/2,074 | Successful: 1,498 | Failed: 2 | Elapsed: 11.0 minutes
Processed 1,550/2,074 | Successful: 1,548 | Failed: 2 | Elapsed: 11.4 minutes
Processed 1,600/2,074 | Successful: 1,598 | Failed: 2 | Elapsed: 11.7 minutes
Processed 1,650/2,074 | Successful: 1,648 | Failed: 2 | Elapsed: 12.1 minutes


...

Processed 2,000/2,074 | Successful: 1,982 | Failed: 18 | Elapsed: 15.1 minutes


$3324.TW: No data found, symbol may be delisted
$3357.TW: No data found, symbol may be delisted
$3362.TW: No data found, symbol may be delisted
$3363.TW: No data found, symbol may be delisted
$3374.TW: No data found, symbol may be delisted


Processed 2,050/2,074 | Successful: 2,027 | Failed: 23 | Elapsed: 15.6 minutes


$3491.TW: No data found, symbol may be delisted
$3526.TW: No data found, symbol may be delisted
$3551.TW: No data found, symbol may be delisted


Processed 2,074/2,074 | Successful: 2,048 | Failed: 26 | Elapsed: 15.9 minutes

Download complete.
Saved to: data\price_data\batch1_prices.parquet
Tickers with price data: 2,048
Trading dates: 2,041
Failed tickers: 26
Total elapsed time: 15.9 minutes


In [18]:

def download_batch(batch_number, period="2y", interval="1d"):

    # Validate batch number (1 to number of batches)
    if not 1 <= batch_number <= len(batches):
        raise ValueError(f"Choose a batch between 1 and {len(batches)}.")

    batch = batches[batch_number - 1]

    print(f"Starting Batch {batch_number}: {len(batch):,} tickers")

    start_time = time.monotonic()
    price_series = []
    failed_tickers = []

    # Download prices one ticker at a time
    for i, ticker in enumerate(batch, start=1):
        try:
            history = yf.Ticker(ticker).history(
                period=period,
                interval=interval,
                auto_adjust=True,
                timeout=30
            )

            if not history.empty and history["Close"].notna().any():
                prices = history["Close"].dropna().rename(ticker)

                # Normalise timestamps before combining exchanges
                prices.index = prices.index.normalize()

                if prices.index.has_duplicates:
                    failed_tickers.append(ticker)
                else:
                    price_series.append(prices)
            else:
                failed_tickers.append(ticker)

        except Exception:
            failed_tickers.append(ticker)

        # Print progress every 50 tickers and at the end
        if i % 50 == 0 or i == len(batch):
            elapsed = (time.monotonic() - start_time) / 60

            print(
                f"Processed {i:,}/{len(batch):,} | "
                f"Successful: {len(price_series):,} | "
                f"Failed: {len(failed_tickers):,} | "
                f"Elapsed: {elapsed:.1f} minutes"
            )

    # Check whether any data was retrieved
    if not price_series:
        print("No price data was retrieved. No price file was saved.")
        return None

    # Combine all successfully retrieved price histories
    prices_df = pd.concat(price_series, axis=1).sort_index()
    prices_df.index.name = "Date"

    # Create output folder
    output_dir = Path("data/price_data")
    output_dir.mkdir(parents=True, exist_ok=True)

    # Save price data
    prices_path = output_dir / f"batch{batch_number}_prices.parquet"
    prices_df.to_parquet(prices_path)

    # Save the failed ticker list
    failed_path = output_dir / f"batch{batch_number}_failed_tickers.csv"
    pd.DataFrame({"Ticker": failed_tickers}).to_csv(
        failed_path, index=False
    )

    # Print final summary
    elapsed = (time.monotonic() - start_time) / 60

    print(f"\nBatch {batch_number} complete.")
    print(f"Price data saved: {prices_path}")
    print(f"Failed ticker list saved: {failed_path}")
    print(f"Tickers with price data: {prices_df.shape[1]:,}")
    print(f"Failed tickers: {len(failed_tickers):,}")
    print(f"Trading dates: {prices_df.shape[0]:,}")
    print(f"Total elapsed time: {elapsed:.1f} minutes")

    return prices_df, failed_tickers


In [19]:
batch2_prices, batch2_failed = download_batch(2)

Starting Batch 2: 2,074 tickers


$3587.TW: No data found, symbol may be delisted
$3675.TW: No data found, symbol may be delisted
$3680.TW: No data found, symbol may be delisted
$3691.TW: No data found, symbol may be delisted
$3693.TW: No data found, symbol may be delisted
$3707.TW: No data found, symbol may be delisted
$3718.TW: No data found, symbol may be delisted


Processed 50/2,074 | Successful: 43 | Failed: 7 | Elapsed: 0.6 minutes
Processed 100/2,074 | Successful: 93 | Failed: 7 | Elapsed: 0.9 minutes


$4105.TW: No data found, symbol may be delisted
$4114.TW: No data found, symbol may be delisted
$4123.TW: No data found, symbol may be delisted
$4147.TW: No data found, symbol may be delisted
$4162.TW: No data found, symbol may be delisted


Processed 150/2,074 | Successful: 138 | Failed: 12 | Elapsed: 1.5 minutes


...

Processed 2,000/2,074 | Successful: 1,893 | Failed: 107 | Elapsed: 17.9 minutes


$ABMB.KL: No data found, symbol may be delisted
$AC.PS: No data found, symbol may be delisted
$ACEN.PS: No data found, symbol may be delisted


Processed 2,050/2,074 | Successful: 1,940 | Failed: 110 | Elapsed: 18.4 minutes


$ACSM.KL: No data found, symbol may be delisted
$ADCB.AE: No data found, symbol may be delisted


Processed 2,074/2,074 | Successful: 1,962 | Failed: 112 | Elapsed: 18.6 minutes

Batch 2 complete.
Price data saved: data\price_data\batch2_prices.parquet
Failed ticker list saved: data\price_data\batch2_failed_tickers.csv
Tickers with price data: 1,962
Failed tickers: 112
Trading dates: 5,893
Total elapsed time: 18.7 minutes


In [20]:
batch3_prices, batch3_failed = download_batch(3)

Starting Batch 3: 2,074 tickers


$ADES.SR: No data found, symbol may be delisted
$ADIB.AE: No data found, symbol may be delisted
$ADNH.AE: No data found, symbol may be delisted
$ADNHC.AE: No data found, symbol may be delisted
$ADNOCDIS.AE: No data found, symbol may be delisted
$ADNOCDRI.AE: No data found, symbol may be delisted
$ADNOCLS.AE: No data found, symbol may be delisted
$ADPORTS.AE: No data found, symbol may be delisted
$ADSB.AE: No data found, symbol may be delisted
$AEM.SI: No data found, symbol may be delisted
$AEON.KL: No data found, symbol may be delisted


Processed 50/2,074 | Successful: 39 | Failed: 11 | Elapsed: 0.7 minutes


$AGI.PS: No data found, symbol may be delisted
$AGILITY.AE: No data found, symbol may be delisted
$AGTHIA.AE: No data found, symbol may be delisted
$AGUILCPO.MX: No data found, symbol may be delisted


Processed 100/2,074 | Successful: 85 | Failed: 15 | Elapsed: 1.2 minutes


...

Processed 2,000/2,074 | Successful: 1,836 | Failed: 164 | Elapsed: 19.4 minutes


$EXTRA.SR: No data found, symbol may be delisted
$FAKEEHCA.SR: No data found, symbol may be delisted
$FCR-U.TO: No data found, symbol may be delisted
$FCT.SI: No data found, symbol may be delisted
$FEH.SI: No data found, symbol may be delisted


Processed 2,050/2,074 | Successful: 1,881 | Failed: 169 | Elapsed: 19.9 minutes


$FEHT.SI: No data found, symbol may be delisted
$FERTIGLB.AE: No data found, symbol may be delisted
$FFB.KL: No data found, symbol may be delisted
$FGEN.PS: No data found, symbol may be delisted


Processed 2,074/2,074 | Successful: 1,901 | Failed: 173 | Elapsed: 20.2 minutes

Batch 3 complete.
Price data saved: data\price_data\batch3_prices.parquet
Failed ticker list saved: data\price_data\batch3_failed_tickers.csv
Tickers with price data: 1,901
Failed tickers: 173
Trading dates: 5,884
Total elapsed time: 20.2 minutes


In [21]:
batch4_prices, batch4_failed = download_batch(4)

Starting Batch 4: 2,073 tickers


$FIBRAMQ.MX: No data found, symbol may be delisted
$FIBRAPL.MX: No data found, symbol may be delisted
$FIRSTMIL.SR: No data found, symbol may be delisted
$FLT.SI: No data found, symbol may be delisted
$FLYNAS.SR: No data found, symbol may be delisted


Processed 50/2,073 | Successful: 45 | Failed: 5 | Elapsed: 0.5 minutes


$FNH.KL: No data found, symbol may be delisted
$FOURTHMI.SR: No data found, symbol may be delisted
$FR.SI: No data found, symbol may be delisted
$FRCB.KL: No data found, symbol may be delisted


Processed 100/2,073 | Successful: 91 | Failed: 9 | Elapsed: 1.1 minutes


...

Processed 2,000/2,073 | Successful: 1,839 | Failed: 161 | Elapsed: 19.7 minutes


$ORH.KL: No data found, symbol may be delisted
$OSK.KL: No data found, symbol may be delisted


Processed 2,050/2,073 | Successful: 1,887 | Failed: 163 | Elapsed: 20.1 minutes


$OUEREIT.SI: No data found, symbol may be delisted


Processed 2,073/2,073 | Successful: 1,909 | Failed: 164 | Elapsed: 20.3 minutes

Batch 4 complete.
Price data saved: data\price_data\batch4_prices.parquet
Failed ticker list saved: data\price_data\batch4_failed_tickers.csv
Tickers with price data: 1,909
Failed tickers: 164
Trading dates: 5,469
Total elapsed time: 20.3 minutes


In [22]:
batch5_prices, batch5_failed = download_batch(5)

Starting Batch 5: 2,073 tickers


$PAD.KL: No data found, symbol may be delisted
$PAG.NS: No data found, symbol may be delisted
$PCHEM.KL: No data found, symbol may be delisted
$PCPG.SI: No data found, symbol may be delisted


Processed 50/2,073 | Successful: 46 | Failed: 4 | Elapsed: 0.5 minutes


$PENT.KL: No data found, symbol may be delisted
$PEP.KL: No data found, symbol may be delisted
$PETD.KL: No data found, symbol may be delisted
$PETROR.SR: No data found, symbol may be delisted
$PFGRUPOA.CL: No data found, symbol may be delisted
$PFGRUPSU.CL: No data found, symbol may be delisted


Processed 100/2,073 | Successful: 90 | Failed: 10 | Elapsed: 1.0 minutes


...

Processed 1,950/2,073 | Successful: 1,786 | Failed: 164 | Elapsed: 19.3 minutes
Processed 2,000/2,073 | Successful: 1,836 | Failed: 164 | Elapsed: 19.6 minutes


$YACCO.SR: No data found, symbol may be delisted
$YAMAMAH.SR: No data found, symbol may be delisted
$YANSAB.SR: No data found, symbol may be delisted
$YLLG.SI: No data found, symbol may be delisted
$YNCCO.SR: No data found, symbol may be delisted
$YNS.KL: No data found, symbol may be delisted
$YTL.KL: No data found, symbol may be delisted
$YTLP.KL: No data found, symbol may be delisted
$YTLREIT.KL: No data found, symbol may be delisted
$YZJSGD.SI: No data found, symbol may be delisted
$Z.NS: No data found, symbol may be delisted
$ZAINKSA.SR: No data found, symbol may be delisted


Processed 2,050/2,073 | Successful: 1,874 | Failed: 176 | Elapsed: 20.3 minutes


$ZETRIX.KL: No data found, symbol may be delisted
$ZIIC.SR: No data found, symbol may be delisted
$ZOUJAJ.SR: No data found, symbol may be delisted


Processed 2,073/2,073 | Successful: 1,894 | Failed: 179 | Elapsed: 20.6 minutes

Batch 5 complete.
Price data saved: data\price_data\batch5_prices.parquet
Failed ticker list saved: data\price_data\batch5_failed_tickers.csv
Tickers with price data: 1,894
Failed tickers: 179
Trading dates: 5,885
Total elapsed time: 20.6 minutes


In [23]:
# Load Batch 3 price data
batch3_prices = pd.read_parquet("data/price_data/batch3_prices.parquet")

# Get the 200th ticker from the Parquet columns
ticker = batch3_prices.columns[199]

print("Ticker:", ticker)
print(batch3_prices[ticker].dropna())

Ticker: AMAGI.NS
Date
2026-01-20 18:30:00+00:00    348.250000
2026-01-21 18:30:00+00:00    365.600006
2026-01-22 18:30:00+00:00    375.649994
2026-01-26 18:30:00+00:00    360.100006
2026-01-27 18:30:00+00:00    366.850006
                                ...    
2026-10-04 18:30:00+00:00    569.700012
2026-10-05 18:30:00+00:00    564.150024
2026-10-06 18:30:00+00:00    561.450012
2026-10-07 18:30:00+00:00    555.299988
2026-10-08 18:30:00+00:00    559.400024
Name: AMAGI.NS, Length: 177, dtype: float64


In [24]:

from pathlib import Path
import pandas as pd

# Load all five batches and count observations per ticker
observation_counts = {}

for batch_number in range(1, 6):
    path = Path(f"data/price_data/batch{batch_number}_prices.parquet")
    prices = pd.read_parquet(path)

    observation_counts.update(prices.notna().sum().to_dict())

# Count how many tickers have each history length
length_distribution = (
    pd.Series(observation_counts, name="Observations")
    .value_counts()
    .sort_index(ascending=False)
)

length_distribution.index.name = "Price observations"
length_distribution.name = "Number of tickers"

print(length_distribution.to_string())
print(f"\nTotal tickers: {len(observation_counts):,}")


Price observations
511     277
509       7
508     235
507     383
506     106
505       2
504       5
503     374
502      36
501    2243
500     313
499     230
498      41
497     647
496     151
495      14
494      19
493     598
492      28
491       1
490       6
489       1
487     993
486    2169
485     123
484       2
483       1
479       3
478       1
477       1
476       1
475     115
474       2
473       4
472       4
471       2
470       1
469       1
467       1
466       1
465       1
464       1
463       2
462       1
461       1
459       2
458       1
457       2
454       2
453       1
452       1
450       1
449       3
447       1
446       1
445       1
444       1
443       4
442       1
441       3
439       1
438       3
435       3
433       1
432       2
430       2
429       1
428       1
427     120
425       1
423       2
422       1
421       1
420       2
419       3
418       1
416      23
415       1
414       3
413       1
412       2
410      

In [9]:
print("Earliest date:", batch1_prices.index.min())
print("Latest date:", batch1_prices.index.max())
print("Number of unique dates:", batch1_prices.index.nunique())

display(batch1_prices.head())
display(batch1_prices.tail())

Earliest date: 2024-10-07 15:00:00+00:00
Latest date: 2026-10-08 22:00:00+00:00
Number of unique dates: 2041


,000001.SZ,000002.SZ,000009.SZ,000021.SZ,000027.SZ,000029.SZ,000032.SZ,000034.SZ,000039.SZ,000050.SZ,...,3545.TW,3549.T,3563.T,3563.TW,356860.KQ,3569.T,3576.TW,357780.KQ,3583.TW,358570.KQ
Date,,,,,,,,,,,,,,,,,,,,,
2024-10-07 15:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,5773.334961,NaN,NaN,211250.203125,NaN,10611.206055
2024-10-07 16:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,82.335793,NaN,NaN,360.796387,NaN,NaN,12.25,NaN,494.966278,NaN
2024-10-07 21:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024-10-08 15:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,3412.554199,1482.031128,NaN,NaN,2518.221436,NaN,NaN,NaN,NaN
2024-10-08 16:00:00+00:00,10.287009,9.34,9.55002,16.866804,6.836331,13.980663,18.766617,23.360651,8.246969,8.17,...,80.802185,NaN,NaN,350.098907,NaN,NaN,11.55,NaN,487.629822,NaN


,000001.SZ,000002.SZ,000009.SZ,000021.SZ,000027.SZ,000029.SZ,000032.SZ,000034.SZ,000039.SZ,000050.SZ,...,3545.TW,3549.T,3563.T,3563.TW,356860.KQ,3569.T,3576.TW,357780.KQ,3583.TW,358570.KQ
Date,,,,,,,,,,,,,,,,,,,,,
2026-10-07 21:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-10-07 22:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-10-08 15:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,4138.0,5454.0,NaN,NaN,3495.0,NaN,NaN,NaN,NaN
2026-10-08 16:00:00+00:00,11.59,4.06,7.0,32.029999,6.34,21.299999,12.64,22.25,8.43,6.23,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-10-08 22:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
# Compare the current index with calendar dates
print("Original unique timestamps:", batch1_prices.index.nunique())
print(
    "Unique UTC calendar dates:",
    batch1_prices.index.tz_convert("UTC").date.__class__
    if False else len(set(batch1_prices.index.tz_convert("UTC").date))
)

# Show the most common timestamp times
print("\nMost common UTC times:")
display(
    pd.Series(batch1_prices.index.strftime("%H:%M:%S %Z"))
    .value_counts()
    .head(10)
)

Original unique timestamps: 2041
Unique UTC calendar dates: 622

Most common UTC times:


Date
15:00:00 UTC    517
16:00:00 UTC    516
21:00:00 UTC    501
22:00:00 UTC    297
23:00:00 UTC    210
Name: count, dtype: int64

In [11]:
# Check whether normalising timestamps to UTC dates creates duplicate dates
normalised_dates = pd.Index(
    batch1_prices.index.tz_convert("UTC").date
)

print("Rows:", len(batch1_prices))
print("Unique UTC dates:", normalised_dates.nunique())
print("Duplicate rows after date normalisation:", normalised_dates.duplicated().sum())

Rows: 2041
Unique UTC dates: 622
Duplicate rows after date normalisation: 1419


In [12]:
# Inspect timestamps for a few stocks
for ticker in batch1_prices.columns[:5]:
    prices = batch1_prices[ticker].dropna()

    print(f"\n{ticker}")
    print("Number of observations:", len(prices))
    print("First timestamps:")
    print(prices.index[:5].tolist())


000001.SZ
Number of observations: 486
First timestamps:
[Timestamp('2024-10-08 16:00:00+0000', tz='UTC'), Timestamp('2024-10-09 16:00:00+0000', tz='UTC'), Timestamp('2024-10-10 16:00:00+0000', tz='UTC'), Timestamp('2024-10-13 16:00:00+0000', tz='UTC'), Timestamp('2024-10-14 16:00:00+0000', tz='UTC')]

000002.SZ
Number of observations: 486
First timestamps:
[Timestamp('2024-10-08 16:00:00+0000', tz='UTC'), Timestamp('2024-10-09 16:00:00+0000', tz='UTC'), Timestamp('2024-10-10 16:00:00+0000', tz='UTC'), Timestamp('2024-10-13 16:00:00+0000', tz='UTC'), Timestamp('2024-10-14 16:00:00+0000', tz='UTC')]

000009.SZ
Number of observations: 486
First timestamps:
[Timestamp('2024-10-08 16:00:00+0000', tz='UTC'), Timestamp('2024-10-09 16:00:00+0000', tz='UTC'), Timestamp('2024-10-10 16:00:00+0000', tz='UTC'), Timestamp('2024-10-13 16:00:00+0000', tz='UTC'), Timestamp('2024-10-14 16:00:00+0000', tz='UTC')]

000021.SZ
Number of observations: 486
First timestamps:
[Timestamp('2024-10-08 16:00:00+00

In [13]:
# Convert each stock's index to its UTC calendar date
price_series = []

for ticker in batch1_prices.columns:
    prices = batch1_prices[ticker].dropna().copy()

    prices.index = prices.index.normalize()

    # Check that this stock has no duplicate dates
    if prices.index.has_duplicates:
        print(f"Duplicate dates found: {ticker}")
        continue

    prices.name = ticker
    price_series.append(prices)

# Recombine all stocks using the normalised dates
batch1_prices = pd.concat(price_series, axis=1).sort_index()
batch1_prices.index.name = "Date"

print("Shape:", batch1_prices.shape)
print("First date:", batch1_prices.index.min())
print("Last date:", batch1_prices.index.max())
print("Unique dates:", batch1_prices.index.nunique())

Shape: (622, 2048)
First date: 2024-10-07 00:00:00+00:00
Last date: 2026-10-08 00:00:00+00:00
Unique dates: 622


In [14]:
# Check missing prices after aligning dates
missing = batch1_prices.isna().sum()

print("Total price observations:", batch1_prices.size)
print("Missing price observations:", batch1_prices.isna().sum().sum())
print("Percentage missing:", round(batch1_prices.isna().sum().sum()
                                   / batch1_prices.size * 100, 2), "%")

print("\nStocks with the most missing observations:")
print(missing.sort_values(ascending=False).head(10))

Total price observations: 1273856
Missing price observations: 300328
Percentage missing: 23.58 %

Stocks with the most missing observations:
2915.HK      621
0668.HK      621
0501.HK      621
0699.HK      621
0638.HK      621
0220WL.KS    593
0220W0.KS    593
3308.HK      571
2249.HK      557
1377.HK      556
dtype: int64


In [15]:
# Inspect the stocks with the most missing prices
worst_tickers = missing.sort_values(ascending=False).head(10).index

for ticker in worst_tickers:
    prices = batch1_prices[ticker].dropna()

    print(
        f"{ticker}: {len(prices)} observations | "
        f"First: {prices.index.min()} | "
        f"Last: {prices.index.max()}"
    )

2915.HK: 1 observations | First: 2026-10-08 00:00:00+00:00 | Last: 2026-10-08 00:00:00+00:00
0668.HK: 1 observations | First: 2026-10-08 00:00:00+00:00 | Last: 2026-10-08 00:00:00+00:00
0501.HK: 1 observations | First: 2026-10-08 00:00:00+00:00 | Last: 2026-10-08 00:00:00+00:00
0699.HK: 1 observations | First: 2026-10-08 00:00:00+00:00 | Last: 2026-10-08 00:00:00+00:00
0638.HK: 1 observations | First: 2026-10-08 00:00:00+00:00 | Last: 2026-10-08 00:00:00+00:00
0220WL.KS: 29 observations | First: 2026-08-25 00:00:00+00:00 | Last: 2026-10-07 00:00:00+00:00
0220W0.KS: 29 observations | First: 2026-08-25 00:00:00+00:00 | Last: 2026-10-07 00:00:00+00:00
3308.HK: 51 observations | First: 2026-07-29 00:00:00+00:00 | Last: 2026-10-08 00:00:00+00:00
2249.HK: 65 observations | First: 2026-07-09 00:00:00+00:00 | Last: 2026-10-08 00:00:00+00:00
1377.HK: 66 observations | First: 2026-07-08 00:00:00+00:00 | Last: 2026-10-08 00:00:00+00:00


In [16]:
# Number of available observations per stock
observations = batch1_prices.notna().sum()

print("Stocks by history length:")
print(f"At least 400 observations: {(observations >= 400).sum():,}")
print(f"200–399 observations: {((observations >= 200) & (observations < 400)).sum():,}")
print(f"50–199 observations: {((observations >= 50) & (observations < 200)).sum():,}")
print(f"Fewer than 50 observations: {(observations < 50).sum():,}")

print("\nStocks with fewer than 200 observations:")
print(observations[observations < 200].sort_values().head(20))

Stocks by history length:
At least 400 observations: 1,962
200–399 observations: 44
50–199 observations: 35
Fewer than 50 observations: 7

Stocks with fewer than 200 observations:
0699.HK        1
0501.HK        1
0638.HK        1
0668.HK        1
2915.HK        1
0220W0.KS     29
0220WL.KS     29
3308.HK       51
2249.HK       65
2475.HK       66
1377.HK       66
1688.HK       74
2493.HK      105
1879.HK      114
3296.HK      117
2476.HK      119
0068.HK      121
3277.HK      121
2726.HK      132
301682.SZ    132
dtype: int64
